# 11 · Graphs — BFS, DFS, Topological Sort and Dijkstra

After this notebook you can store a graph two ways, pick the right traversal for the question (BFS, DFS, Kahn's topological sort, Dijkstra), and detect cycles in directed and undirected graphs.

**Time:** ~45 min · **Runs:** offline on CPU in < 1 min · **Needs:** [Stacks and queues](04_stacks_and_queues.ipynb), [Recursion and backtracking](08_recursion_and_backtracking.ipynb), [Heaps and top-k](10_heaps_and_top_k.ipynb)

## Why this matters in interviews

- Many problems are graphs in disguise: grids (islands, rotting oranges), dependencies (course schedule, build order), word transformations (word ladder), networks (delay time). The first step is to say "this is a graph: the nodes are ..., the edges are ...".
- Four algorithms cover almost everything: **BFS** (fewest steps), **DFS** (reachability, components, cycles), **topological sort** (an order that respects dependencies), **Dijkstra** (cheapest path with non-negative weights).

## What interviewers ask

- "Number of islands" (LeetCode 200), "rotting oranges" (994), "flood fill" (733).
- "Course schedule — can you finish? in what order?" (207 / 210) — topological sort.
- "Shortest path in an unweighted graph / word ladder" (127) — BFS.
- "Network delay time / cheapest route" (743) — Dijkstra.
- "Detect a cycle", "is the graph bipartite?" (785), "clone a graph" (133).
- Follow-ups: "adjacency list or matrix, and why?", "why does BFS give the shortest path?", "what breaks Dijkstra?", "how do you detect a cycle in a *directed* graph?"

In [ ]:
import heapq
import math
import random
from collections import defaultdict, deque

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy import ndimage

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "axes.axisbelow": True, "axes.titlesize": 11,
                     "legend.frameon": False})
BLUE, ORANGE, AQUA, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#52514e"

def draw_graph(pos, edges, ax=None, title=None, directed=False, node_colors=None,
               hot_edges=(), weights=None, hot_color=ORANGE):
    """Manual-layout graph drawing. pos: {node: (x, y)}; hot_edges are drawn thick in hot_color."""
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(5.5, 3.4))
    hot = set(hot_edges) | ({(v, u) for u, v in hot_edges} if not directed else set())
    for u, v in edges:
        (x1, y1), (x2, y2) = pos[u], pos[v]
        color, lw = (hot_color, 2.6) if (u, v) in hot else ("#b5b3ac", 1.4)
        if directed:
            ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(
                arrowstyle="-|>", color=color, lw=lw, shrinkA=13, shrinkB=13, mutation_scale=14))
        else:
            ax.plot([x1, x2], [y1, y2], color=color, lw=lw, zorder=1)
        if weights:
            ax.text((x1 + x2) / 2, (y1 + y2) / 2, str(weights[(u, v)]), fontsize=9, color=MUTED,
                    ha="center", va="center", bbox=dict(fc="white", ec="none", pad=0.6), zorder=2)
    for node, (x, y) in pos.items():
        fc = (node_colors or {}).get(node, "#e7f0ff")
        ax.scatter([x], [y], s=520, color=fc, edgecolors=BLUE, linewidths=1.5, zorder=3)
        ax.text(x, y, str(node), ha="center", va="center", fontsize=10, zorder=4)
    xs, ys = zip(*pos.values())
    ax.set_xlim(min(xs) - 0.6, max(xs) + 0.6); ax.set_ylim(min(ys) - 0.6, max(ys) + 0.6)
    ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10)
    if own:
        plt.show()

## 1. The pattern

**In plain English:** a graph is a set of things (**nodes**) and connections between them (**edges**). Edges can be one-way (**directed**) or two-way (**undirected**), and can carry a cost (**weighted**). A tree is a graph with no cycles; a graph may have cycles, so every traversal needs a **visited** set, or it loops forever.

| Question in the problem | Algorithm | Frontier data structure |
|---|---|---|
| fewest steps / shortest path, all edges equal | BFS | queue (`deque`) |
| can I reach it? how many groups? is there a cycle? | DFS | stack / recursion |
| an order that respects "A before B" | topological sort (Kahn) | queue of nodes with in-degree 0 |
| cheapest path, non-negative weights | Dijkstra | min-heap on distance |

### Representations: adjacency list vs adjacency matrix

In [ ]:
edges = [("A", "B"), ("A", "C"), ("B", "D"), ("C", "D"), ("D", "E"), ("E", "F")]
nodes = sorted({x for e in edges for x in e})

adj = defaultdict(list)                       # adjacency list: node -> neighbours
for u, v in edges:
    adj[u].append(v); adj[v].append(u)        # undirected: store both directions

index = {n: i for i, n in enumerate(nodes)}
matrix = np.zeros((len(nodes), len(nodes)), dtype=int)    # adjacency matrix: V x V
for u, v in edges:
    matrix[index[u], index[v]] = matrix[index[v], index[u]] = 1

print("adjacency list:", dict(adj))
assert (matrix == matrix.T).all()                                    # undirected -> symmetric
assert [int(matrix[index[n]].sum()) for n in nodes] == [len(adj[n]) for n in nodes]   # degrees agree

small_pos = {"A": (0, 1), "B": (1, 2), "C": (1, 0), "D": (2, 1), "E": (3, 1), "F": (4, 1)}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw={"width_ratios": [1.5, 1]})
draw_graph(small_pos, edges, axes[0], "the graph")
axes[1].imshow(matrix, cmap="Blues", vmin=0, vmax=1.6)
axes[1].set_xticks(range(len(nodes)), nodes); axes[1].set_yticks(range(len(nodes)), nodes)
axes[1].grid(False)
for i in range(len(nodes)):
    for j in range(len(nodes)):
        axes[1].text(j, i, matrix[i, j], ha="center", va="center", fontsize=9,
                     color="white" if matrix[i, j] else MUTED)
axes[1].set_title("adjacency matrix (row -> column)", fontsize=10)
plt.show()

| | Adjacency list | Adjacency matrix |
|---|---|---|
| memory | O(V + E) | O(V²) |
| "is there an edge u–v?" | O(degree) (O(1) with sets) | O(1) |
| iterate the neighbours of u | O(degree) | O(V) |
| best for | sparse graphs — almost every interview graph | dense graphs, fast edge checks, matrix maths |

Default to a `defaultdict(list)` adjacency list. Grids are graphs too, with implicit edges to the 4 neighbours — no need to build anything.

## 2. BFS: shortest path when every edge costs the same

**In plain English:** BFS explores in rings — everything 1 step away, then everything 2 steps away, and so on. The first time it reaches a node, it has used the fewest possible steps, so BFS gives shortest paths in unweighted graphs. Keep a `parent` map to rebuild the path.

**The rule that avoids duplicate work:** mark a node as visited when you **enqueue** it, not when you pop it.

**Brute force for comparison:** enumerate every simple path from start to goal (exponential) and keep the shortest.

In [ ]:
bfs_edges = [(0, 1), (0, 2), (1, 3), (2, 3), (2, 4), (3, 5), (4, 6), (5, 7), (6, 7), (7, 8), (4, 9), (9, 8)]
bfs_adj = defaultdict(list)
for u, v in bfs_edges:
    bfs_adj[u].append(v); bfs_adj[v].append(u)

def bfs_shortest_path(adj, start, goal):
    parent, queue = {start: None}, deque([start])
    while queue:
        node = queue.popleft()
        if node == goal:
            break
        for nb in adj[node]:
            if nb not in parent:              # visited-on-enqueue
                parent[nb] = node
                queue.append(nb)
    if goal not in parent:
        return None
    path = []
    while goal is not None:
        path.append(goal); goal = parent[goal]
    return path[::-1]

def bfs_distances(adj, start):
    dist, queue = {start: 0}, deque([start])
    while queue:
        node = queue.popleft()
        for nb in adj[node]:
            if nb not in dist:
                dist[nb] = dist[node] + 1
                queue.append(nb)
    return dist

def all_simple_paths(adj, start, goal, path=None):          # brute force: exponential in general
    path = path or [start]
    if path[-1] == goal:
        yield path
        return
    for nb in adj[path[-1]]:
        if nb not in path:
            yield from all_simple_paths(adj, start, goal, path + [nb])

path = bfs_shortest_path(bfs_adj, 0, 8)
dist = bfs_distances(bfs_adj, 0)
brute = min(all_simple_paths(bfs_adj, 0, 8), key=len)
print("BFS path 0 -> 8:", path, "| length:", len(path) - 1)
print("distances from 0:", dict(sorted(dist.items())))
assert len(path) - 1 == dist[8] == len(brute) - 1 == 4
assert bfs_shortest_path(bfs_adj, 0, 42) is None

In [ ]:
bfs_pos = {0: (0, 1), 1: (1, 2), 2: (1, 0), 3: (2, 2), 4: (2, 0), 5: (3, 2), 6: (3, 0.7),
           7: (4, 1.6), 8: (5, 1), 9: (3.6, -0.3)}
ring_colors = plt.cm.Blues(np.linspace(0.12, 0.75, max(dist.values()) + 1))
colors = {n: ring_colors[d] for n, d in dist.items()}
draw_graph(bfs_pos, bfs_edges, node_colors=colors, hot_edges=list(zip(path, path[1:])),
           title="BFS rings: darker = more steps from node 0; orange = a shortest path to 8")

**Complexity:** O(V + E) time, O(V) space. Every node is enqueued once, every edge looked at once (twice if undirected).

## 3. DFS: go deep, then backtrack

**In plain English:** DFS follows one path as far as it can, then backs up and tries the next branch — the backtracking walk of [notebook 08](08_recursion_and_backtracking.ipynb) with a visited set. It answers "can I reach it?", "how many connected pieces?", "is there a cycle?". It does **not** give shortest paths.

Recursive DFS is the shortest code; iterative DFS (an explicit stack) survives deep graphs. Pushing neighbours in reverse makes the iterative version visit in the same order as the recursive one.

In [ ]:
def dfs_recursive(adj, start):
    order, seen = [], set()
    def visit(u):
        seen.add(u); order.append(u)
        for v in adj[u]:
            if v not in seen:
                visit(v)
    visit(start)
    return order

def dfs_iterative(adj, start):
    order, seen, stack = [], set(), [start]
    while stack:
        u = stack.pop()
        if u in seen:
            continue
        seen.add(u); order.append(u)
        stack.extend(v for v in reversed(adj[u]) if v not in seen)
    return order

def count_components(nodes, adj):
    seen, count = set(), 0
    for n in nodes:
        if n not in seen:
            count += 1
            seen.update(dfs_iterative(adj, n))
    return count

print("DFS order from 0:", dfs_recursive(bfs_adj, 0))
assert dfs_recursive(bfs_adj, 0) == dfs_iterative(bfs_adj, 0)
rnd = random.Random(0)
for _ in range(200):
    n = rnd.randint(1, 15)
    g = defaultdict(list)
    for _ in range(rnd.randint(0, 25)):
        u, v = rnd.randrange(n), rnd.randrange(n)
        if u != v and v not in g[u]:
            g[u].append(v); g[v].append(u)
    assert dfs_recursive(g, 0) == dfs_iterative(g, 0)
    assert set(dfs_iterative(g, 0)) == set(bfs_distances(g, 0))          # same reachable set as BFS
print("iterative == recursive order, and DFS reaches exactly what BFS reaches, on 200 random graphs")
print("components in {A..F} + an isolated 'G':", count_components(nodes + ["G"], adj))

### Problem — Number of islands (LeetCode 200)

A grid of land (1) and water (0); an island is land connected up/down/left/right. Count the islands. Scan every cell; each time you meet unvisited land, that is a new island — flood-fill it (DFS or BFS) so its other cells are not counted again. Iterative flood fill, because a big island would blow the recursion limit. `scipy.ndimage.label` does the same thing (4-connectivity by default) and makes an independent check.

In [ ]:
DIRS = ((1, 0), (-1, 0), (0, 1), (0, -1))

def num_islands(grid):
    rows, cols = len(grid), len(grid[0])
    label = [[0] * cols for _ in range(rows)]
    count = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 1 and label[r][c] == 0:
                count += 1                                   # unvisited land: a new island
                label[r][c] = count
                stack = [(r, c)]
                while stack:                                 # flood fill it
                    i, j = stack.pop()
                    for di, dj in DIRS:
                        ni, nj = i + di, j + dj
                        if 0 <= ni < rows and 0 <= nj < cols and grid[ni][nj] == 1 and label[ni][nj] == 0:
                            label[ni][nj] = count
                            stack.append((ni, nj))
    return count, label

classic = [[1, 1, 0, 0, 0], [1, 1, 0, 0, 0], [0, 0, 1, 0, 0], [0, 0, 0, 1, 1]]
assert num_islands(classic)[0] == 3 and num_islands([[0]])[0] == 0 and num_islands([[1]])[0] == 1

grng = np.random.default_rng(3)
grid = (grng.random((10, 18)) < 0.45).astype(int).tolist()
count, label = num_islands(grid)
assert count == ndimage.label(np.array(grid))[1]
for seed in range(50):
    g = (np.random.default_rng(seed).random((15, 15)) < 0.5).astype(int)
    assert num_islands(g.tolist())[0] == ndimage.label(g)[1]
print(f"random 10x18 grid: {count} islands (scipy.ndimage.label agrees; so it does on 50 more grids)")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.imshow(np.array(grid), cmap=ListedColormap(["#f0efec", "#86b6ef"]))
first_cell = {}
for r in range(len(grid)):
    for c in range(len(grid[0])):
        if label[r][c] and label[r][c] not in first_cell:
            first_cell[label[r][c]] = (r, c)
for island, (r, c) in first_cell.items():
    ax.text(c, r, island, ha="center", va="center", fontsize=8, fontweight="bold", color=INK)
ax.axis("off")
ax.set_title(f"{count} islands (4-connected land); numbers mark where the scan first met each one")
plt.show()

**Complexity:** O(R · C) time — each cell is labelled once — and O(R · C) space in the worst case (the stack for one huge island).

## 4. Cycle detection

- **Undirected:** during DFS, reaching an already-visited node that is **not the parent** you came from means there are two routes to it: a cycle. (Union-find does it too — [notebook 14](14_tries_union_find_lru_cache.ipynb).)
- **Directed:** "visited" is not enough. Use three colours: **white** (unseen), **grey** (on the current DFS path), **black** (finished). An edge to a **grey** node is a back edge — a cycle. An edge to a black node is fine: that node was fully explored from another branch.

The classic trap is the diamond `A→B, A→C, B→D, C→D`: D is reached twice, but there is no directed cycle.

In [ ]:
def has_cycle_undirected(nodes, edges):
    adj = defaultdict(list)
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    seen = set()
    for s in nodes:
        if s in seen:
            continue
        seen.add(s)
        stack = [(s, None)]
        while stack:
            u, parent = stack.pop()
            for v in adj[u]:
                if v == parent:
                    continue                       # the edge we arrived on
                if v in seen:
                    return True                    # a second route to v
                seen.add(v); stack.append((v, u))
    return False

def has_cycle_directed_naive(nodes, adj):         # WRONG: treats "seen before" as a cycle
    seen = set()
    def visit(u):
        seen.add(u)
        return any(v in seen or visit(v) for v in adj[u])
    return any(n not in seen and visit(n) for n in nodes)

WHITE, GREY, BLACK = 0, 1, 2
def has_cycle_directed(nodes, adj):
    color = {n: WHITE for n in nodes}
    def visit(u):
        color[u] = GREY                            # u is on the current path
        for v in adj[u]:
            if color[v] == GREY:
                return True                        # back edge -> cycle
            if color[v] == WHITE and visit(v):
                return True
        color[u] = BLACK                           # finished: no cycle through u
        return False
    return any(color[n] == WHITE and visit(n) for n in nodes)

diamond = {"A": ["B", "C"], "B": ["D"], "C": ["D"], "D": []}
loop3 = {"A": ["B"], "B": ["C"], "C": ["A"]}
print("diamond  -> naive:", has_cycle_directed_naive(diamond, diamond), "| three-colour:", has_cycle_directed(diamond, diamond))
print("A->B->C->A -> three-colour:", has_cycle_directed(loop3, loop3))
assert has_cycle_directed_naive(diamond, diamond) and not has_cycle_directed(diamond, diamond)
assert has_cycle_directed(loop3, loop3)
assert has_cycle_undirected("ABC", [("A", "B"), ("B", "C"), ("C", "A")])
assert not has_cycle_undirected("ABCD", [("A", "B"), ("B", "C"), ("B", "D")])

## 5. Topological sort — Kahn's algorithm

**In plain English:** you have tasks and "A must happen before B" rules. Repeatedly take any task with no unfinished prerequisites (in-degree 0), do it, and cross it off its dependants' lists. If you get stuck before doing every task, the leftover tasks form a cycle, and no valid order exists.

Processing the zero-in-degree tasks **in waves** also tells you which tasks can run in parallel — the same thing a DAG scheduler (Airflow, `make -j`) exploits.

In [ ]:
def kahn_levels(graph):
    """graph: {task: [tasks that depend on it]}. Returns waves of tasks; raises if there is a cycle."""
    indeg = {n: 0 for n in graph}
    for u in graph:
        for v in graph[u]:
            indeg[v] += 1
    wave, levels = sorted(n for n in graph if indeg[n] == 0), []
    while wave:
        levels.append(wave)
        nxt = []
        for u in wave:
            for v in graph[u]:
                indeg[v] -= 1
                if indeg[v] == 0:
                    nxt.append(v)
        wave = sorted(nxt)
    if sum(map(len, levels)) < len(graph):
        raise ValueError(f"cycle: no valid order for {sorted(n for n in graph if indeg[n] > 0)}")
    return levels

def topo_order(graph):
    return [task for wave in kahn_levels(graph) for task in wave]

pipeline = {                                   # a build-and-release pipeline
    "checkout": ["install_deps"], "install_deps": ["compile"], "compile": ["build_image", "unit_tests"],
    "build_image": ["deploy_staging"], "deploy_staging": ["e2e_tests"], "unit_tests": ["e2e_tests"],
    "load_fixtures": ["seed_database"], "seed_database": ["e2e_tests"], "e2e_tests": ["release"],
    "release": [],
}
levels = kahn_levels(pipeline)
order = topo_order(pipeline)
for i, wave in enumerate(levels):
    print(f"wave {i}: {wave}")
position = {t: i for i, t in enumerate(order)}
assert all(position[u] < position[v] for u in pipeline for v in pipeline[u])   # every edge points forward
assert levels[0] == ["checkout", "load_fixtures"] and levels[-1] == ["release"]

In [ ]:
lane = {"checkout": 0.6, "install_deps": 0.6, "compile": 0.6, "unit_tests": 1.0, "build_image": 0.2,
        "deploy_staging": 0.2, "e2e_tests": -0.2, "release": -0.2, "load_fixtures": -0.8, "seed_database": -0.8}
wave_of = {task: x for x, wave in enumerate(levels) for task in wave}
fig, ax = plt.subplots(figsize=(11, 3.4))
boxes = {task: ax.text(wave_of[task] * 1.6, lane[task], task, ha="center", va="center", fontsize=8.5,
                       bbox=dict(boxstyle="round,pad=0.4", fc="#e7f0ff", ec=BLUE)) for task in pipeline}
for u in pipeline:
    for v in pipeline[u]:
        ax.annotate("", xy=boxes[v].get_position(), xytext=boxes[u].get_position(),
                    arrowprops=dict(arrowstyle="-|>", color="#8a887f", lw=1.2, shrinkA=3, shrinkB=3,
                                    patchA=boxes[u].get_bbox_patch(), patchB=boxes[v].get_bbox_patch()))
for x in range(len(levels)):
    ax.text(x * 1.6, -1.3, f"wave {x}", ha="center", fontsize=8, color=MUTED)
ax.set_xlim(-0.8, (len(levels) - 1) * 1.6 + 0.8); ax.set_ylim(-1.4, 1.3); ax.axis("off")
ax.set_title("The pipeline DAG laid out by Kahn waves: tasks in the same column can run in parallel")
plt.show()

**Complexity:** O(V + E). A cycle is reported, not looped on:

In [ ]:
feedback = {k: list(v) for k, v in pipeline.items()}
feedback["e2e_tests"].append("compile")          # "recompile if the tests fail" creates a cycle
try:
    kahn_levels(feedback)
    raise AssertionError("expected a cycle")
except ValueError as e:
    print("ValueError:", e)

def can_finish(num_courses, prerequisites):         # LeetCode 207 is exactly this
    graph = {i: [] for i in range(num_courses)}
    for course, pre in prerequisites:
        graph[pre].append(course)
    try:
        kahn_levels(graph)
        return True
    except ValueError:
        return False

assert can_finish(2, [[1, 0]]) and not can_finish(2, [[1, 0], [0, 1]]) and can_finish(1, [])
print("course schedule: [[1,0]] ->", can_finish(2, [[1, 0]]), "| [[1,0],[0,1]] ->", can_finish(2, [[1, 0], [0, 1]]))

## 6. Dijkstra: cheapest path with non-negative weights

**In plain English:** BFS with a priority queue. Always expand the unfinished node with the smallest known distance; because weights are never negative, nothing found later can make that node cheaper, so its distance is final. Push `(distance, node)` into a heap; when you pop a stale entry for a node that is already final, skip it (lazy deletion).

**Brute force baseline:** Bellman-Ford — relax every edge V − 1 times, O(V · E). It is slower but also handles negative weights, which makes it the reference answer.

In [ ]:
def dijkstra(graph, source):
    """graph: {u: [(v, w), ...]} with every w >= 0. Returns (dist, parent)."""
    dist, parent = {source: 0}, {source: None}
    heap, done = [(0, source)], set()
    while heap:
        d, u = heapq.heappop(heap)
        if u in done:
            continue                                    # stale heap entry
        done.add(u)                                     # d is now final for u
        for v, w in graph[u]:
            nd = d + w
            if v not in done and nd < dist.get(v, math.inf):
                dist[v], parent[v] = nd, u
                heapq.heappush(heap, (nd, v))
    return dist, parent

def bellman_ford(nodes, weighted_edges, source):
    dist = {n: math.inf for n in nodes}
    dist[source] = 0
    for _ in range(len(nodes) - 1):
        for u, v, w in weighted_edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
    return dist

road = [("S", "A", 4), ("S", "B", 1), ("B", "A", 2), ("A", "C", 1), ("B", "D", 5), ("C", "D", 1), ("C", "T", 6), ("D", "T", 2)]
road_graph = defaultdict(list)
for u, v, w in road:                                     # undirected roads
    road_graph[u].append((v, w)); road_graph[v].append((u, w))
dist, parent = dijkstra(road_graph, "S")
route, node = [], "T"
while node is not None:
    route.append(node); node = parent[node]
route = route[::-1]
print("cheapest S -> T:", " -> ".join(route), "| cost:", dist["T"])
both_ways = road + [(v, u, w) for u, v, w in road]
assert dist == bellman_ford(set(road_graph), both_ways, "S")

for _ in range(200):                                     # random directed graphs, non-negative weights
    n = rnd.randint(1, 12)
    wedges = [(rnd.randrange(n), rnd.randrange(n), rnd.randint(0, 9)) for _ in range(rnd.randint(0, 30))]
    g = defaultdict(list)
    for u, v, w in wedges:
        g[u].append((v, w))
    d, _ = dijkstra(g, 0)
    bf = bellman_ford(range(n), wedges, 0)
    assert d == {k: v for k, v in bf.items() if v < math.inf}
print("Dijkstra == Bellman-Ford on 200 random graphs")

In [ ]:
road_pos = {"S": (0, 1), "A": (1.5, 2), "B": (1.5, 0), "C": (3, 2), "D": (3, 0), "T": (4.5, 1)}
draw_graph(road_pos, [(u, v) for u, v, _ in road], weights={(u, v): w for u, v, w in road},
           hot_edges=list(zip(route, route[1:])),
           title=f"Dijkstra: cheapest S -> T is {' -> '.join(route)} (cost {dist['T']}), not the fewest hops")

**Complexity:** O((V + E) log V) with a binary heap; O(V + E) space.

**The trap: negative weights.** Dijkstra declares a node final when it is popped. With a negative edge, a cheaper route can appear later, after the node — and everything downstream of it — was settled.

In [ ]:
neg_edges = [("S", "A", 1), ("S", "B", 2), ("B", "A", -2), ("A", "C", 1)]
neg_graph = defaultdict(list)
for u, v, w in neg_edges:
    neg_graph[u].append((v, w))
dj, _ = dijkstra(neg_graph, "S")
bf = bellman_ford({"S", "A", "B", "C"}, neg_edges, "S")
print(f"distance to C -> Dijkstra: {dj['C']}  | Bellman-Ford: {bf['C']}  (true route S -> B -> A -> C)")
assert dj["C"] == 2 and bf["C"] == 1

## 7. Bipartite check (LeetCode 785)

Can the nodes be split into two groups with every edge going **between** the groups? BFS and colour alternately; an edge between two nodes of the same colour means an **odd cycle**, and the answer is no. (Users ↔ items, students ↔ courses are naturally bipartite graphs.)

In [ ]:
def is_bipartite(adj, nodes):
    color = {}
    for s in nodes:
        if s in color:
            continue
        color[s] = 0
        queue = deque([s])
        while queue:
            u = queue.popleft()
            for v in adj[u]:
                if v not in color:
                    color[v] = 1 - color[u]
                    queue.append(v)
                elif color[v] == color[u]:
                    return False, color, (u, v)          # same colour on both ends: odd cycle
    return True, color, None

def to_adj(edge_list):
    g = defaultdict(list)
    for u, v in edge_list:
        g[u].append(v); g[v].append(u)
    return g

square = [(0, 1), (1, 2), (2, 3), (3, 0), (1, 4)]
pentagon = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0)]
ok_sq, col_sq, _ = is_bipartite(to_adj(square), range(5))
ok_pe, col_pe, clash = is_bipartite(to_adj(pentagon), range(5))
print("square + tail:", ok_sq, "| pentagon (odd cycle):", ok_pe, "- clash on edge", clash)
lc_false = {i: nb for i, nb in enumerate([[1, 2, 3], [0, 2], [0, 1, 3], [0, 2]])}
lc_true = {i: nb for i, nb in enumerate([[1, 3], [0, 2], [1, 3], [0, 2]])}
assert ok_sq and not ok_pe and not is_bipartite(lc_false, range(4))[0] and is_bipartite(lc_true, range(4))[0]

side = {0: "#cde2fb", 1: "#fde7dc"}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
draw_graph({0: (0, 1), 1: (1, 1), 2: (1, 0), 3: (0, 0), 4: (2, 1)}, square, axes[0],
           "bipartite: every edge joins blue to peach", node_colors={n: side[c] for n, c in col_sq.items()})
pent_pos = {i: (math.cos(math.pi / 2 + 2 * math.pi * i / 5), math.sin(math.pi / 2 + 2 * math.pi * i / 5)) for i in range(5)}
draw_graph(pent_pos, pentagon, axes[1], "odd cycle: the dark edge joins two same-colour nodes",
           node_colors={n: side[c] for n, c in col_pe.items()}, hot_edges=[clash], hot_color=INK)
plt.show()

**Complexity:** O(V + E). Loop over all start nodes, or a disconnected graph is only half checked.

## Try it yourself

**1. Rotting oranges (LeetCode 994).** `2` = rotten, `1` = fresh, `0` = empty. Each minute, rot spreads to 4-neighbours. Minutes until no fresh orange remains, or `-1` if some never rot. Hint: **multi-source BFS** — start the queue with every rotten orange at once.

In [ ]:
# Solution — try it yourself first, then run this
def oranges_rotting(grid):
    rows, cols = len(grid), len(grid[0])
    grid = [row[:] for row in grid]
    queue = deque((r, c) for r in range(rows) for c in range(cols) if grid[r][c] == 2)
    fresh = sum(row.count(1) for row in grid)
    minutes = 0
    while queue and fresh:
        for _ in range(len(queue)):                    # one BFS ring = one minute
            r, c = queue.popleft()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1:
                    grid[nr][nc] = 2; fresh -= 1; queue.append((nr, nc))
        minutes += 1
    return minutes if fresh == 0 else -1

assert oranges_rotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]) == 4
assert oranges_rotting([[2, 1, 1], [0, 1, 1], [1, 0, 1]]) == -1
assert oranges_rotting([[0, 2]]) == 0
print("minutes:", oranges_rotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]))

**2. Word ladder (LeetCode 127).** Transform `begin` into `end` changing one letter at a time, every intermediate word in the list. Return the number of words in the shortest sequence (0 if impossible). Hint: BFS on an implicit graph; bucket words by patterns like `h*t` so neighbours are found without comparing every pair.

In [ ]:
# Solution — try it yourself first, then run this
def ladder_length(begin, end, words):
    words = set(words)
    if end not in words:
        return 0
    buckets = defaultdict(list)
    for w in words | {begin}:
        for i in range(len(w)):
            buckets[w[:i] + "*" + w[i + 1:]].append(w)
    seen, queue = {begin}, deque([(begin, 1)])
    while queue:
        word, steps = queue.popleft()
        if word == end:
            return steps
        for i in range(len(word)):
            for nb in buckets[word[:i] + "*" + word[i + 1:]]:
                if nb not in seen:
                    seen.add(nb); queue.append((nb, steps + 1))
    return 0

assert ladder_length("hit", "cog", ["hot", "dot", "dog", "lot", "log", "cog"]) == 5
assert ladder_length("hit", "cog", ["hot", "dot", "dog", "lot", "log"]) == 0
print("hit -> cog:", ladder_length("hit", "cog", ["hot", "dot", "dog", "lot", "log", "cog"]), "words")

**3. Network delay time (LeetCode 743).** A signal leaves node `k`; `times[i] = (u, v, w)` is a directed edge taking `w`. How long until all `n` nodes (numbered 1..n) have it, or `-1` if some never do? Hint: Dijkstra, then take the maximum distance.

In [ ]:
# Solution — try it yourself first, then run this
def network_delay_time(times, n, k):
    graph = defaultdict(list)
    for u, v, w in times:
        graph[u].append((v, w))
    dist, _ = dijkstra(graph, k)
    return max(dist.values()) if len(dist) == n else -1

assert network_delay_time([(2, 1, 1), (2, 3, 1), (3, 4, 1)], 4, 2) == 2
assert network_delay_time([(1, 2, 1)], 2, 1) == 1 and network_delay_time([(1, 2, 1)], 2, 2) == -1
print("delay:", network_delay_time([(2, 1, 1), (2, 3, 1), (3, 4, 1)], 4, 2))

## Say it in an interview

**30-second answer:** "First I say what the nodes and edges are. For fewest steps with equal edge costs I use BFS with a queue, marking nodes visited when I enqueue them — O(V + E). For reachability, components or cycles, DFS. For an order that respects dependencies, Kahn's algorithm: repeatedly take nodes with in-degree zero; if some are left over, there is a cycle. For weighted shortest paths with non-negative weights, Dijkstra with a min-heap, O((V + E) log V); with negative weights, Bellman-Ford."

**Follow-ups and traps:**
- *BFS visited-on-pop* works but can enqueue the same node many times; mark on enqueue.
- *Directed cycles* need three colours; "visited before" is wrong (the diamond).
- *Dijkstra with negative edges* silently returns wrong answers; skip stale heap entries.
- *Grids:* iterate instead of recursing (a 1000×1000 island is a million-deep recursion).
- *Disconnected graphs:* loop over every start node for components, bipartite and cycle checks.

## Pattern cheat sheet

| Signal | Algorithm | Structure | Time |
|---|---|---|---|
| fewest steps, unweighted, "minimum moves", word ladder | BFS (multi-source if many starts) | `deque` + visited | O(V + E) |
| connected, count groups, flood fill, islands | DFS / BFS / union-find | stack or queue | O(V + E) |
| cycle, undirected | DFS with parent (or union-find) | visited set | O(V + E) |
| cycle, directed | DFS three colours (or Kahn leftovers) | colour map | O(V + E) |
| prerequisites, build order, pipeline scheduling | Kahn's topological sort | in-degree + queue | O(V + E) |
| cheapest path, weights ≥ 0 | Dijkstra | min-heap `(dist, node)` | O((V + E) log V) |
| negative weights | Bellman-Ford | edge list | O(V · E) |
| two groups, odd cycle | BFS 2-colouring | colour map | O(V + E) |

## Next

- [14 · Tries, union-find and LRU cache](14_tries_union_find_lru_cache.ipynb) — union-find for connectivity and cycles.
- [12 · Dynamic programming](12_dynamic_programming.ipynb) — shortest paths on a DAG are DP.
- Theory: [DSA basics course — graphs chapter](../../dsa_basics/index.html)